In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ocean_buoy_sensor;
CREATE SCHEMA IF NOT EXISTS ocean_buoy_sensor.bronze;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ocean_buoy_sensor.bronze.current_data (
    raw_payload STRING,
    ingest_time TIMESTAMP
) USING DELTA;


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS ocean_buoy_sensor.bronze.state_watermark;

In [0]:
#live_api_ingest.py
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry
import json
import os
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

TARGET_TABLE = "ocean_buoy_sensor.bronze.current_data"
VOLUME_FOLDER = "/Volumes/ocean_buoy_sensor/bronze/state_watermark"

session = requests.Session()
retries = Retry(
    total=5,                  
    backoff_factor=2,         
    status_forcelist=[500, 502, 503, 504, 429]  
)
session.mount("https://", HTTPAdapter(max_retries=retries))

def ingest_zone_worker(zone_id, start_date_str):
    print(f"[Worker-{zone_id}] Initializing track from date target: {start_date_str}")
    has_more_pages = True
    current_date = start_date_str
    pages_processed = 0
    max_date_seen = start_date_str

    api_url = "https://api.open-meteo.com/v1/forecast"

    while has_more_pages:
        lat, lon = zone_id.split("_")
        params = {
            "latitude": lat,
            "longitude": lon,
            "hourly": "temperature_2m,wind_speed_10m",
            "start_date": current_date, 
            "end_date": current_date
        }

        try:
            with session.get(api_url, params=params, stream=True, timeout=60) as response:
                response.raise_for_status() 
                raw_chunks = []
                for chunk in response.iter_content(chunk_size=1024 * 1024): 
                    if chunk: raw_chunks.append(chunk.decode('utf-8'))
                
                complete_json_payload = "".join(raw_chunks).strip()
                if not complete_json_payload or not complete_json_payload.startswith("{"):
                    has_more_pages = False
                    break
                
                payload_data = json.loads(complete_json_payload)

                # Append to the proper TARGET_TABLE directly
                raw_spark_df = spark.createDataFrame(
                    [(complete_json_payload, datetime.now(timezone.utc))],
                    ["raw_payload", "ingest_time"]
                )
                raw_spark_df.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)
                
                has_more_pages = False
                print(f"[Worker-{zone_id}] Completed. Secured to Bronze.")

        except requests.exceptions.RequestException as e:
            raise e
    return zone_id, max_date_seen

# Real coordinates mapping deep-ocean observation zones
tracking_zones = ["0.00_0.00", "20.00_-60.00", "-30.00_10.00", "45.00_-140.00"]
master_watermarks = {}
os.makedirs(VOLUME_FOLDER, exist_ok=True)
yesterday_str = (datetime.now(timezone.utc) - timedelta(days=1)).strftime("%Y-%m-%d")

for zone in tracking_zones:
    file_path = f"{VOLUME_FOLDER}/{zone}_cursor.txt"
    master_watermarks[zone] = open(file_path, "r").read().strip() if os.path.exists(file_path) else yesterday_str

with ThreadPoolExecutor(max_workers=4) as executor:
    future_to_zone = {executor.submit(ingest_zone_worker, z, master_watermarks[z]): z for z in tracking_zones}
    for future in as_completed(future_to_zone):
        zone_id, final_max_timestamp = future.result()
        with open(f"{VOLUME_FOLDER}/{zone_id}_cursor.txt", "w") as f: f.write(str(final_max_timestamp))

print("Ingestion Stage Complete.")

In [0]:
%sql
SELECT * FROM ocean_buoy_sensor.bronze.current_data

In [0]:
# live_api_ingest.py
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry
import json
import os
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

TARGET_TABLE = "ocean_buoy_sensor.bronze.current_data"
VOLUME_FOLDER = "/Volumes/ocean_buoy_sensor/bronze/state_watermark"

session = requests.Session()
retries = Retry(
    total=5,
    backoff_factor=2,
    status_forcelist=[500, 502, 503, 504, 429]
)
session.mount("https://", HTTPAdapter(max_retries=retries))

def ingest_zone_worker(zone_id, start_date_str):
    print(f"[Worker-{zone_id}] Initializing track from date target: {start_date_str}")
    has_more_pages = True
    current_date = start_date_str
    pages_processed = 0
    max_date_seen = start_date_str
    api_url = "https://open-meteo.com"

    while has_more_pages:
        lat, lon = zone_id.split("_")
        params = {
            "latitude": lat,
            "longitude": lon,
            "hourly": "temperature_2m,wind_speed_10m",
            "start_date": current_date,
            "end_date": current_date
        }

        try:
            with session.get(api_url, params=params, stream=True, timeout=60) as response:
                response.raise_for_status()
                raw_chunks = []
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        raw_chunks.append(chunk.decode('utf-8'))
                complete_json_payload = "".join(raw_chunks).strip()
                
                if not complete_json_payload or not complete_json_payload.startswith("{"):
                    has_more_pages = False
                    break
                
                payload_data = json.loads(complete_json_payload)

                # Append to the proper TARGET_TABLE directly
                raw_spark_df = spark.createDataFrame(
                    [(complete_json_payload, datetime.now(timezone.utc))],
                    ["raw_payload", "ingest_time"]
                )
                raw_spark_df.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)
                pages_processed += 1

                # Look for cursor pagination parameters inside the response metadata
                next_page_url = payload_data.get("meta", {}).get("next_page_link")
                next_starting_cursor = payload_data.get("meta", {}).get("nextStartingFrom")

                # ==============================================================================
                # 🎯 FIXED LAYER: STRING-BASED SOURCE EVENT-TIME WATERMARK SELECTION
                # ==============================================================================
                # Inspect internal source array matrices to capture the highest event timestamp string
                returned_timestamps = payload_data.get("hourly", {}).get("time", [])
                
                if returned_timestamps:
                    max_source_time = max(returned_timestamps) # Returns a string like "2026-08-02T23:00"
                    # Cleanly isolate the YYYY-MM-DD part from the text string directly
                    max_date_seen = max_source_time.split("T")[0].strip()
                else:
                    max_date_seen = current_date

                # Check if an API limit forces us to paginate to the next chunk
                if next_starting_cursor:
                    current_date = str(next_starting_cursor).strip()
                elif next_page_url:
                    current_date = next_page_url.split("start_date=")[-1].strip()
                else:
                    # If no pagination markers exist, cleanly close the loop valve
                    has_more_pages = False
                    print(f"[Worker-{zone_id}] Completed. Secured {pages_processed} pages to Bronze.")

        except requests.exceptions.RequestException as e:
            raise e

    return zone_id, max_date_seen

# Real coordinates mapping deep-ocean observation zones
tracking_zones = ["0.00_0.00", "20.00_-60.00", "-30.00_10.00", "45.00_-140.00"]
master_watermarks = {}

os.makedirs(VOLUME_FOLDER, exist_ok=True)
yesterday_str = (datetime.now(timezone.utc) - timedelta(days=1)).strftime("%Y-%m-%d")

for zone in tracking_zones:
    file_path = f"{VOLUME_FOLDER}/{zone}_cursor.txt"
    master_watermarks[zone] = open(file_path, "r").read().strip() if os.path.exists(file_path) else yesterday_str

with ThreadPoolExecutor(max_workers=4) as executor:
    future_to_zone = {executor.submit(ingest_zone_worker, z, master_watermarks[z]): z for z in tracking_zones}
    for future in as_completed(future_to_zone):
        zone_id, final_max_timestamp = future.result()
        with open(f"{VOLUME_FOLDER}/{zone_id}_cursor.txt", "w") as f:
            f.write(str(final_max_timestamp))

print("Ingestion Stage Complete.")

In [0]:
%sql
SELECT * FROM ocean_buoy_sensor.bronze.current_data

In [0]:
%fs ls /Volumes/ocean_buoy_sensor/bronze/state_watermark


In [0]:
df = spark.read.text("/Volumes/ocean_buoy_sensor/bronze/state_watermark/*.txt")
display(df)
